## 📑 Core Payment Order
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AYAMGw)**  
  Comprehensive documentation outlining the business requirements for this report.

#####Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
%python
# Databricks notebook source
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]
 
dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

##### Schema of the table

In [0]:
table_name = 'Core_PaymentOrder'

schema = (
"BusinessDate date,"
"PortfolioName string,"
"AgreementKey string,"
"AgreementKeyClean string,"
"ExposureKey string,"
"CDI string,"
"PartyTaxRegistrationNumber string,"
"DebtorName string,"
"Product string,"
"SubProduct string,"
"AssetSegment string,"
"IsActiveFlag boolean,"
"AccountStatus string,"
"TotalBalanceEur decimal(19,2),"
"PaymentOrderCurrentAgency string,"
"DPStatus string,"
"Source string,"
"DPDate date,"
"RecentEpidosiDate date,"
"AEpidosiDate date,"
"BEpidosiDate date,"
"AuctionCaseFlag string,"
"AuctionCIFFlag string,"
"AuctionSource string,"
"KatathesiFlag string,"
"EkdosiFlag string,"
"AEpidosiFlag string,"
"BEpidosiFlag string,"
"TelesidikiaDPFlag string,"
"ExistingInInternalFile string,"
"Priority string,"
"KatathesiMaxDateQC date,"
"EkdosiMaxDateQC date,"
"AEpidosiMaxDateQC date,"
"BEpidosiMaxDateQC date,"
"InsertTimeStamp timestamp"
)

In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Optimizer function

In [0]:
%python
# Helper function to create or replace table
def create_or_replace_table(table_name, df, partition_col):
    df.write.format("delta").mode("overwrite").partitionBy(partition_col).saveAsTable(table_name)
def create_or_replace_table_no_partition(table_name, df):
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)
def repair_and_analyze_table(table_name):
    spark.sql(f"REPAIR TABLE {table_name} SYNC METADATA")
    spark.sql(f"ANALYZE TABLE {table_name} COMPUTE STATISTICS")    


##### PaymentOrderHistorical

In [0]:
core_paymentorderhistorical = spark.sql(
f'''
select * from `{env_prefix}-reportinputfiles`.repoint.core_paymentorderhistorical
'''
)
core_paymentorderhistorical.createOrReplaceTempView("core_paymentorderhistorical")

##### dexposure

In [0]:
%python
dexposure_active = spark.sql(
f'''
SELECT 
    A.BusinessDate
    ,A.PortfolioName
    ,A.ExposureKey
    , A.AgreementKey
    ,A.IsActiveFlag
    ,A.ProductKey
    ,A.CDI
    ,A.PartyTaxRegistrationNumber
    ,A.AssetSegment
    ,A.ExposureStatusCode
    ,A.OffBalance
    ,A.OnBalance
    ,A.Bucket
    ,A.ClosedDate
    ,a.ClosedReason
    ,A.Currency
    ,a.casecode
    ,a.productkey
FROM `{env_prefix}-cdh`.derivative.dexposure as A
where A.BusinessDate = {run_date!r}
and SourceSystem = 'qca'
and PortfolioName = 'Core Retail'
and IsActiveFlag = 'true'

'''
)
dexposure_active.createOrReplaceTempView("dexposure_active")



dexposure_inactive = spark.sql(
f'''
SELECT
    A.BusinessDate
    ,A.PortfolioName
    ,A.ExposureKey
    , A.AgreementKey
    ,false as IsActiveFlag
    ,A.ProductKey
    ,A.CDI
    ,A.PartyTaxRegistrationNumber
    ,A.AssetSegment
    ,A.ExposureStatusCode
    ,A.OffBalance
    ,A.OnBalance
    ,A.Bucket
    ,A.ClosedDate
    ,a.ClosedReason
    ,A.Currency
    ,a.casecode
    ,a.productkey
from (select e.*
      , rank() over (partition by e.ExposureKey order by e.BusinessDate desc) as Rank
      from `{env_prefix}-cdh`.derivative.dexposure e
      where 1=1
        and  e.SourceSystem = 'qca'
       -- and e.IsActiveFlag = false
        and e.BusinessDate <= {run_date!r}
        and  e.PortfolioName = 'Core Retail'
        ) AS A
left anti join dexposure_active e_pf_act on e_pf_act.ExposureKey = A.ExposureKey and e_pf_act.BusinessDate = {run_date!r}
where A.Rank = 1
'''
)
dexposure_inactive.createOrReplaceTempView("dexposure_inactive")



Dexposure = spark.sql(
f'''
select *
from dexposure_active
 
union all
 
select *
from dexposure_inactive
 
'''
)
Dexposure.createOrReplaceTempView("Dexposure")



Dexposure_CR = spark.sql(
f'''
SELECT  A.BusinessDate
    ,A.PortfolioName
    ,A.ExposureKey
FROM `{env_prefix}-cdh`.derivative.dexposure A
WHERE BusinessDate ='2021-12-21'
AND PortfolioName ='Cosmos Retail'

'''
)
Dexposure_CR.createOrReplaceTempView("Dexposure_CR")


Dexposure_CW = spark.sql(
f'''
SELECT  A.BusinessDate
    ,A.PortfolioName
    ,A.ExposureKey
FROM `{env_prefix}-cdh`.derivative.dexposure A
WHERE BusinessDate = '2021-12-20'
AND PortfolioName='Cosmos Wholesale'

'''
)
Dexposure_CW.createOrReplaceTempView("Dexposure_CW")


Dexposure_GR = spark.sql(
f'''
SELECT  A.BusinessDate
    ,A.PortfolioName
    ,A.ExposureKey
FROM `{env_prefix}-cdh`.derivative.dexposure A
WHERE BusinessDate='2021-06-22'
AND PortfolioName in ('Galaxy II Funding DAC','Orion X Securitisation DAC')

'''
)
Dexposure_GR.createOrReplaceTempView("Dexposure_GR")


Dexposure_GW = spark.sql(
f'''
SELECT  A.BusinessDate
    ,A.PortfolioName
    ,A.ExposureKey
FROM `{env_prefix}-cdh`.derivative.dexposure A
WHERE BusinessDate='2021-06-22'
AND PortfolioName='Galaxy IV Funding DAC'

'''
)
Dexposure_GW.createOrReplaceTempView("Dexposure_GW")



##### Exposure

In [0]:
exposure = spark.sql(
f'''
SELECT 
    A.BusinessDate
    ,A.PortfolioName
    ,A.ExposureKey
    ,a.Bucket
    ,a.PartyName
    ,a.CDI
FROM `{env_prefix}-cdh`.primary.exposure as A
where A.BusinessDate = {run_date!r}
and SourceSystem = 'qca'
and PortfolioName = 'Core Retail'
'''
)
exposure.createOrReplaceTempView("exposure")

##### Product

In [0]:

Product = spark.sql(
f'''
SELECT distinct
    d.exposurekey
    ,p.BankingUnitCode as SubProduct
FROM `{env_prefix}-cdh`.primary.Product p
left join dexposure d on d.ProductKey = p.ProductKey --and p.businessdate = d.BusinessDate
where 1=1
and p.BusinessDate = {run_date!r}
and sourcesystem = 'qca'
'''
)
Product.createOrReplaceTempView("Product")

In [0]:
Product2 = spark.sql(
f'''
select distinct
    ProductKey
    ,SubProduct
    ,BankingUnitCode
from `{env_prefix}-cdh`.primary.product
Where BusinessDate = {run_date!r}
and SourceSystem ='qca'
'''
)
Product2.createOrReplaceTempView("Product2")

##### Party 

In [0]:
Party = spark.sql(
f'''
SELECT DISTINCT
     CDI,
     TaxRegistrationNumber,
     FirstNameGR,
     LastNameGR
FROM (
    SELECT
        CDI,
        TaxRegistrationNumber,
        FirstNameGR,
        LastNameGR,
        BusinessDate,
        ROW_NUMBER() OVER (PARTITION BY CDI ORDER BY BusinessDate DESC) AS rn
    FROM `{env_prefix}-cdh`.derivative.dparty
) sub
WHERE rn = 1;
'''
)
Party.createOrReplaceTempView("Party")

##### NonCorePerimeter

In [0]:
%python
NonCorePerimeter = spark.sql(
f'''
    select distinct CASE 
        WHEN LEFT(Exposurekey, 1) BETWEEN 'A' AND 'Z' THEN  
            SUBSTRING(Exposurekey, 4, LEN(Exposurekey) - 3) 
        ELSE 
            ExposureKey 
    END AS Exposurekey 
    from `{env_prefix}-cdh`.primary.exposure 
    where PortfolioName <> 'Core Retail'
    and BusinessDate <= {run_date!r}
'''
)
NonCorePerimeter.createOrReplaceTempView("NonCorePerimeter")

# create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_NonCorePerimeter", NonCorePerimeter)
# repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_NonCorePerimeter")

##### Casecode 

In [0]:
casecode = spark.sql(
f'''
select distinct d.exposurekey as accountnumber,casecodekey 
from `{env_prefix}-cdh`.primary.casecode c
inner join `{env_prefix}-cdh`.primary.exposure d on c.casecodekey = d.casecode
where d.portfolioname = 'Core Retail'  and c.PortfolioName = 'Core Retail' 
'''
)
casecode.createOrReplaceTempView("casecode")

##### core_paymentorderinteraction

In [0]:
core_paymentorderinteraction = spark.sql(
f'''
select distinct 
 l.ExposureKey 
,l.InteractionKey
,l.InteractionCode
,l.CDI 
,l.PortfolioName 
,l.InteractionDate 
,l.InteractionTypeCode
,l.InteractionTypeKey 
,l.InteractionType
,l.BusinessDate
 FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
where 1=1
and InteractionDate <= {run_date!r}
and l.PortfolioName = 'Core Retail'
and l.SourceSystem='qca'
and InteractionTypeKey in ( '1203' , '1205', '1206', '1207', '1208', '1209', '1210', '1212', '1213', '1214', '1215', '1216', '1221', '1223', '1224', '1225', '1226', '1227', '1230'
												 , '1231', '1232', '1233', '1234', '1235', '1236', '1237', '1238', '1239', '1240', '1241', '1242', '1243', '1244', '1245', '1246', '1248', '1251', '1252', '1254'
												 , '1257', '1261', '1263', '1264', '1265', '1266', '1267', '1289', '1299', '1304', '1305', '1405', '1703', '1704', '1712', '1713', '1714', '1715', '1719', '1720'
												 , '1721', '1722', '1723', '1724', '1725', '1726', '1727', '1729', '1730', '1731', '1732', '1733', '1734', '1735', '1736', '1737', '1738', '1739', '1740', '1741'
												 , '1742', '1743', '1744', '1745', '1746', '1747', '1748', '1749', '1750', '1751', '1752', '1753', '1754', '1755', '1756', '1757', '1758', '1759', '1760', '1761'
												 , '1763', '1764', '1765', '1766', '1767', '1768', '1769', '1770', '1775', '1776', '1778', '1779', '1780', '1781', '1782', '1783', '1784', '1785', '1790', '1791'
												 , '7032', '905', '1253')

union

select distinct 
 c.accountnumber as exposurekey
,l.InteractionKey
,l.InteractionCode
,l.CDI 
,l.PortfolioName 
,l.InteractionDate 
,l.InteractionTypeCode
,l.InteractionTypeKey 
,l.InteractionType
,l.BusinessDate
FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
inner join casecode c on c.casecodekey = l.casecode
where 1=1
and InteractionDate <= {run_date!r}
and l.SourceSystem='qca'
and InteractionTypeKey in ( '1203' , '1205', '1206', '1207', '1208', '1209', '1210', '1212', '1213', '1214', '1215', '1216', '1221', '1223', '1224', '1225', '1226', '1227', '1230'
												 , '1231', '1232', '1233', '1234', '1235', '1236', '1237', '1238', '1239', '1240', '1241', '1242', '1243', '1244', '1245', '1246', '1248', '1251', '1252', '1254'
												 , '1257', '1261', '1263', '1264', '1265', '1266', '1267', '1289', '1299', '1304', '1305', '1405', '1703', '1704', '1712', '1713', '1714', '1715', '1719', '1720'
												 , '1721', '1722', '1723', '1724', '1725', '1726', '1727', '1729', '1730', '1731', '1732', '1733', '1734', '1735', '1736', '1737', '1738', '1739', '1740', '1741'
												 , '1742', '1743', '1744', '1745', '1746', '1747', '1748', '1749', '1750', '1751', '1752', '1753', '1754', '1755', '1756', '1757', '1758', '1759', '1760', '1761'
												 , '1763', '1764', '1765', '1766', '1767', '1768', '1769', '1770', '1775', '1776', '1778', '1779', '1780', '1781', '1782', '1783', '1784', '1785', '1790', '1791'
												 , '7032', '905', '1253')

'''
)
core_paymentorderinteraction.createOrReplaceTempView("core_paymentorderinteraction")

create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_core_paymentorderinteraction", core_paymentorderinteraction)
repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_core_paymentorderinteraction")

##### Agent

In [0]:

agent = spark.sql(
f'''

select DISTINCT cc1.AccountNumber, cc1.CaseCodeKey, cc2.PaymentOrderCurrentAgency 
from casecode cc1 
left join `{env_prefix}-cdh`.primary.casecode cc2
on cc1.casecodeKey = cc2.casecodekey 
Where 1=1 
and cc2.BusinessDate = {run_date!r}
and cc2.PortfolioName = 'Core Retail'
and cc2.SourceSystem='qca'
and cc2.PaymentOrderCurrentAgency is not null
'''
)
agent.createOrReplaceTempView("agent")

##### MaxBusinessDates

In [0]:
%python
MaxBusinessDates = spark.sql(
f'''
select mbb.ExposureKey, mbb.MaxBusinessDate, c.NextBusinessDay NextBusinessDayOfMaxBusinessDate 
from (
select ExposureKey, max(BusinessDate) maxBusinessDate
from `{env_prefix}-cdh`.primary.exposure 
where PortfolioName = 'Core Retail' and IsActiveFlag = 'true'
group by ExposureKey
) mbb
inner join `{env_prefix}-cdh`.reference.ref_calendar c
on mbb.maxBusinessDate = c.CalendarDate 
'''
)
MaxBusinessDates.createOrReplaceTempView("MaxBusinessDates")

##### Perimeter

In [0]:
%python
core_paymentorderperimeter = spark.sql(
f'''
SELECT distinct 
        A.ExposureKey,
       A.AgreementKey,
       CASE 
           WHEN D.SubProduct LIKE '25%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '01%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '23%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '02%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '36%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '37%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '18%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '03%' THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           WHEN D.SubProduct LIKE '%ΣΣΕ84%' THEN cast(A.AgreementKey as string)
           WHEN LEN(A.AgreementKey)=10 THEN CAST(RIGHT(cast(A.AgreementKey as string), LEN(cast(A.AgreementKey as string))-2) AS string)
           ELSE cast(A.AgreementKey as string)
       END AS AgreementkeyClean,
       A.CDI,
       CASE WHEN A.PartyTaxRegistrationNumber IN ('999999999','000000000','0000000000','') THEN NULL ELSE A.PartyTaxRegistrationNumber END AS PartyTaxRegistrationNumber,
       CASE WHEN P.SubProduct = 'CRED'       THEN '02'
        	WHEN P.SubProduct = 'AMEX'       THEN '01'
			WHEN P.SubProduct = 'ACBDYN'     THEN '18'
			WHEN P.SubProduct = 'ACBST'      THEN '03'
			WHEN P.SubProduct = 'ACBCL'      THEN '25'
			WHEN P.SubProduct = 'AMXPFS'     THEN '23'
			WHEN P.SubProduct = 'CREDC'      THEN '36'
			WHEN P.SubProduct = 'DINERS'     THEN '37'
			ELSE P.SubProduct 
   		END AS Product,
        D.SubProduct as SubProduct,
        A.IsActiveFlag as IsActiveFlag,
        A.AssetSegment AS AssetSegment,
        A.ExposureStatusCode as ArrearsStatus,
        A.Bucket as SourceBucket,
        CONCAT(dp.FirstNameGR, ' ', dp.LastNameGR) as DebtorName
        ,CASE WHEN A.IsActiveFlag=true then
            CASE WHEN A.ExposureStatusCode='074-WRITE OFF'	
                THEN ROUND(`{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.OffBalance, A.BusinessDate), 2) ELSE ROUND(`{env_prefix}-cdh`.derivative.converttoeur(A.Currency, A.OffBalance + A.OnBalance, A.BusinessDate), 2)
            END
		END  as TotalBalanceEur,
        c.PaymentOrderCurrentAgency as PaymentOrderCurrentAgency,
        a.closeddate as ClosingDate,
        a.ClosedReason as ClosingReason,
        CASE WHEN a.IsActiveFlag =false 					THEN 'Κλειστός Λογαριασμός'
					 WHEN a.ExposureStatusCode='074-WRITE OFF'		THEN 'Σε Διαγραφή'
					 WHEN a.ExposureStatusCode='074-DENOUNCED'		THEN 'Σε Καταγγελία'
					 												ELSE 'Σε Καθυστέρηση'
				END AS AccountStatus
from Dexposure A
LEFT JOIN Product D ON A.ExposureKey = D.ExposureKey
left join Product2 P on A.ProductKey = P.ProductKey
left join party as dp on dp.CDI = A.CDI
left join agent c on c.AccountNumber = A.ExposureKey 
'''
)
core_paymentorderperimeter.createOrReplaceTempView("core_paymentorderperimeter")

# create_or_replace_table_no_partition(f"`{env_prefix}-cdh-delivery`.repoint.int_core_paymentorderperimeter", Perimeter)
# repair_and_analyze_table(f"`{env_prefix}-cdh-delivery`.repoint.int_core_paymentorderperimeter")


##### AllActivities

In [0]:
%python
Activities = spark.sql(
f'''
select distinct 
 l.ExposureKey
,l.InteractionKey
,l.CDI
,l.PortfolioName
,l.InteractionDate as ActivityDate
, l.InteractionTypeKey as ActivityTypeKey
,l.InteractionType as ActivityName
,l.BusinessDate
from `{env_prefix}-cdh-delivery`.repoint.int_core_paymentorderinteraction l
left join MaxBusinessDates as mbb 
on mbb.ExposureKey = l.ExposureKey
WHERE 1=1
and l.InteractionDate <= ifnull(last_day(mbb.NextBusinessDayOfMaxBusinessDate), '9999-12-30') 
'''
)
Activities.createOrReplaceTempView("Activities")

In [0]:
%python
AllActivities = spark.sql(
f'''
SELECT  distinct
					{run_date!r}		AS BusinessDate
				,	LA.InteractionKey
				,	LA.ExposureKey
				,	LA.CDI
				,	LA.PortfolioName
				,	CAST (LA.ActivityDate AS DATE) AS ActivityDate
				,	LA.ActivityTypeKey
				,	LA.ActivityName
				--,	LA.TypeOfActivity
				,	CASE WHEN LA.ActivityTypeKey LIKE '17%'								THEN 'Auctions'
						 WHEN LA.ActivityTypeKey = '1203'								THEN 'Katathesi'
						 WHEN LA.ActivityTypeKey IN ('1205', '1235', '905')				THEN 'Ekdosi'
						 WHEN LA.ActivityTypeKey IN ('1206', '1207', '1243', '1263')	THEN 'AEpidosi'
						 WHEN LA.ActivityTypeKey IN ('1212', '1213', '1214', '1266')	THEN 'BEpidosi'
						 WHEN LA.ActivityTypeKey = '1230'								THEN 'Telesidikia'
					END AS ActivityCategory
				,	CASE WHEN LA.ActivityTypeKey LIKE '17%'								THEN 6  --'Auctions'
						 WHEN LA.ActivityTypeKey = '1203'								THEN 1	--'Katathesi'
						 WHEN LA.ActivityTypeKey IN ('1205', '1235', '905')				THEN 2	--'Ekdosi'
						 WHEN LA.ActivityTypeKey IN ('1206', '1207', '1243', '1263')	THEN 3	--'AEpidosi'
						 WHEN LA.ActivityTypeKey IN ('1212', '1213', '1214', '1266')	THEN 4	--'BEpidosi'
						 WHEN LA.ActivityTypeKey = '1230'								THEN 5	--'Telesidikia'
					END AS FlagsRank
				,	CASE WHEN LA.ActivityTypeKey NOT IN ('1203', '1205', '1235', '905') THEN 1
																							ELSE 0
					END AS RestActivity
from Activities LA
'''
)
AllActivities.createOrReplaceTempView("AllActivities")

#####PaymentOrder

In [0]:
%python
PaymentOrder = spark.sql(
f'''
select 
                {run_date!r} as BusinessDate
                ,   h.Account_Number																		AS ExposureKey
                ,	'Core Retail' AS PortfolioName
                ,	IF(ACC.ExposureKey IS NOT NULL,ACC.CDI,H.CIF_GROUP)					AS CDI
                ,	IF(ACC.ExposureKey IS NOT NULL, ACC.AgreementKey,H.Contract_Number)		AS AgreementKey
                ,	IF(ACC.ExposureKey IS NOT NULL, ACC.AgreementKeyClean,H.Contract_Number )	AS AgreementKeyClean
                ,	IF(ACC.ExposureKey IS NOT NULL, ACC.PartyTaxRegistrationNumber, H.AFM)						AS PartyTaxRegistrationNumber
                ,	IF(ACC.ExposureKey IS NOT NULL, ACC.DebtorName, H.Debtor_Name_Calculated)	AS DebtorName
                ,	IF(ACC.ExposureKey IS NOT NULL, ACC.Product, H.PRODUCT)						AS Product
                ,	IF(ACC.ExposureKey IS NOT NULL, ACC.Subproduct, H.Sub_Product)				AS SubProduct
                ,	IF(ACC.ExposureKey IS NOT NULL, ACC.AssetSegment, H.Business_Unit)			AS AssetSegment
                ,	IF(ACC.ExposureKey IS NOT NULL, ACC.IsActiveFlag, false)								AS IsActiveFlag
                ,	IF(ACC.ExposureKey IS NOT NULL, ACC.AccountStatus, 'Κλειστός Λογαριασμός')		AS AccountStatus
                ,  CAST( IF(ACC.IsActiveFlag =true , ACC.TotalBalanceEur, REPLACE(H.Total_Balance, ',', '.')) as DOUBLE) AS TotalBalanceEur
                ,	IF(ACC.ExposureKey IS NOT NULL, acc.PaymentOrderCurrentAgency, NULL)				AS PaymentOrderCurrentAgency
                ,	H.DP_STATUS								AS DPStatus
                ,	H.SOURCE								AS Source
                ,	H.DP_Date								AS DPDate
                ,	H.Recent_Epidosi						AS RecentEpidosiDate
                ,	H.A_Epidosi								AS AEpidosiDate
                ,	H.B_Epidosi								AS BEpidosiDate
                ,	H.Auction_Case_Flag						AS AuctionCaseFlag
                ,	H.Auction_CIF_Flag						AS AuctionCIFFlag
                ,	H.Auction_Source						AS AuctionSource
                ,	H.`Flag_Κατάθεση`						AS KatathesiFlag
                ,	H.`Flag_Έκδοση`							AS EkdosiFlag
                ,	H.`Flag_Α_Επίδοση`						AS AEpidosiFlag
                ,	H.`Flag_Β_Επίδοση`						AS BEpidosiFlag
                ,	H.`Flag_Τελεσιδικία_ΔΠ`					AS TelesidikiaDPFlag
                ,	H.Existing_found_in_internal_file		AS ExistingInInternalFile
                ,	H.Priority								AS Priority
from Core_PaymentOrderHistorical H
left join core_paymentorderperimeter ACC on ACC.ExposureKey = H.ACCOUNT_NUMBER
-- left join `{env_prefix}-cdh-delivery`.repoint.int_core_paymentorderperimeter ACC on ACC.ExposureKey = H.ACCOUNT_NUMBER
WHERE 1=1 
              AND (H.SPV_Description_Calculated IS NULL or H.SPV_Description_Calculated = 'GEMINI CORE SECURITISATION DAC')

'''
)
PaymentOrder.createOrReplaceTempView("PaymentOrder")

##### PaymentOrder

In [0]:

%python
step1 = spark.sql(
f'''
SELECT DISTINCT
					{run_date!r} as BusinessDate
				,	E.ExposureKey
				,	E.PortfolioName
				,	A.CDI
				,	A.AgreementKey
				,	A.AgreementKeyClean
				,	A.PartyTaxRegistrationNumber
				,	a.DebtorName
				,	A.Product
				,	A.SubProduct
				,	A.AssetSegment
				,	A.IsActiveFlag
				,	A.AccountStatus
				,	a.TotalBalanceEur
				,	a.PaymentOrderCurrentAgency
                ,   null as DPStatus
                ,   null as Source
                ,   null as DPDate
                ,   null as RecentEpidosiDate
                ,   null as AEpidosiDate
                ,   null as BEpidosiDate
                ,   null as AuctionCaseFlag
                ,   null as AuctionCIFFlag
                ,   null as AuctionSource
                ,   null as KatathesiFlag  
                ,   null as EkdosiFlag  
                ,   null as AEpidosiFlag  
                ,   null as BEpidosiFlag
                ,   null as TelesidikiaDPFlag  
                ,   null as ExistingInInternalFile  
                ,   null as Priority
		FROM AllActivities				AS E
		INNER JOIN core_paymentorderperimeter			    AS A ON A.ExposureKey = E.ExposureKey 
		-- INNER JOIN `{env_prefix}-cdh-delivery`.repoint.int_core_paymentorderperimeter			    AS A ON A.ExposureKey = E.ExposureKey 
		LEFT JOIN PaymentOrder			AS H ON E.ExposureKey = H.ExposureKey	
		WHERE 1=1
			  AND H.ExposureKey IS NULL
			  AND A.Subproduct NOT LIKE '%3109%' AND A.Subproduct NOT LIKE '%3110%' AND A.Subproduct NOT LIKE '%3111%' AND A.Subproduct NOT LIKE '%3112%' AND 
				  A.Subproduct NOT LIKE '%3114%' AND A.Subproduct NOT LIKE '%3115%' AND A.Subproduct NOT LIKE '%3117%' AND A.Subproduct NOT LIKE '%3119%' AND 
				  A.Subproduct NOT LIKE '%3120%' AND A.Subproduct NOT LIKE '%3121%' AND A.Subproduct NOT LIKE '%3122%' AND A.Subproduct NOT LIKE '%3124%' AND
				  A.Subproduct NOT LIKE '%3126%' AND A.Subproduct NOT LIKE '%3127%' AND A.Subproduct NOT LIKE '%3146%' AND A.Subproduct NOT LIKE '%3147%' AND 
				  A.Subproduct NOT LIKE '%3148%' AND A.Subproduct NOT LIKE '%3149%' AND A.Subproduct NOT LIKE '%3152%' AND A.Subproduct NOT LIKE '%40010%'

'''
)
step1.createOrReplaceTempView("step1")
		


##### PaymentOrderNew

In [0]:
%python
PaymentOrder_new = spark.sql(
f'''
select * from PaymentOrder
union
select * from step1
'''
)
PaymentOrder_new.createOrReplaceTempView("PaymentOrder_new")

##### StrongestAct

In [0]:
%python
StrongestAct = spark.sql(
f'''
SELECT 
					ExposureKey				
				,	MAX(CASE WHEN E.RestActivity = 1				THEN 3
							 WHEN E.ActivityCategory = 'Ekdosi'		THEN 2
							 WHEN E.ActivityCategory = 'Katathesi'	THEN 1
						END)	AS StrongestDP
				,	MIN(CASE WHEN E.ActivityCategory IN ('Katathesi', 'Ekdosi') THEN E.ActivityDate 
																					ELSE NULL
						END)	AS MinEkdosiKatathesi
				,	MAX(CASE WHEN E.ActivityCategory = 'AEpidosi'	THEN E.ActivityDate 
																		ELSE NULL
						END)	AS MaxAEpidosi
				,	MAX(CASE WHEN E.ActivityCategory = 'BEpidosi'	THEN E.ActivityDate 
																		ELSE NULL
						END)	AS MaxBEpidosi
				,	MAX(CASE WHEN E.ActivityCategory IN ('BEpidosi', 'AEpidosi')	THEN E.ActivityDate 
																						ELSE NULL
						END)	AS MaxEpidosi
				,	MAX(CASE WHEN E.FlagsRank BETWEEN 1 AND 5	THEN E.FlagsRank
																	ELSE -1
						END)	AS MaxFlagsRankExclAuct
				,	MAX(E.FlagsRank)	AS MaxRank
				,	MAX(CASE WHEN E.ActivityCategory IN ('Katathesi') THEN E.ActivityDate 
																					ELSE NULL
						END)	AS KatathesiMaxDate
				,	MAX(CASE WHEN E.ActivityCategory IN ('Ekdosi') THEN E.ActivityDate 
																					ELSE NULL
						END)	AS EkdosiMaxDate
		FROM AllActivities AS E
		GROUP BY ExposureKey
'''
)
StrongestAct.createOrReplaceTempView("StrongestAct")


##### Update PaymentOrder

In [0]:
%python
Updated = spark.sql(
f'''
select 
   PO.BusinessDate
  ,PO.ExposureKey
  ,PO.PortfolioName
  ,PO.CDI
  ,PO.AgreementKey
  ,PO.AgreementKeyClean
  ,PO.PartyTaxRegistrationNumber
  ,PO.DebtorName
  ,PO.Product
  ,PO.SubProduct
  ,PO.AssetSegment
  ,PO.IsActiveFlag
  ,PO.AccountStatus
  ,PO.TotalBalanceEur
  ,PO.PaymentOrderCurrentAgency
  ,CASE WHEN SA.StrongestDP = 3		THEN 'ΚΟΙΝΟΠΟΙΗΣΗ'
				WHEN SA.StrongestDP = 2		THEN 'ΕΚΔΟΣΗ'
				WHEN SA.StrongestDP = 1		THEN 'ΚΑΤΑΘΕΣΗ'
				ELSE PO.DPStatus					
    END AS DPStatus
    ,'Qualco' AS Source
    ,CASE WHEN PO.DPDate IS NULL	THEN SA.MinEkdosiKatathesi
		  WHEN SA.MinEkdosiKatathesi IS NOT NULL AND 
				PO.DPDate > SA.MinEkdosiKatathesi	THEN SA.MinEkdosiKatathesi
		  ELSE PO.DPDate
	END AS DPDate
    ,CASE WHEN PO.RecentEpidosiDate IS NULL	THEN SA.MaxEpidosi
		  WHEN SA.MaxEpidosi IS NOT NULL AND 
			   PO.RecentEpidosiDate < SA.MaxEpidosi	THEN SA.MaxEpidosi
		  ELSE PO.RecentEpidosiDate
	END AS RecentEpidosiDate
    ,CASE WHEN PO.AEpidosiDate IS NULL					THEN SA.MaxAEpidosi
	      WHEN SA.MaxAEpidosi IS NOT NULL AND 
			   PO.AEpidosiDate < SA.MaxAEpidosi		THEN SA.MaxAEpidosi
	      ELSE PO.AEpidosiDate
	END AS AEpidosiDate
    ,CASE WHEN PO.BEpidosiDate IS NULL	THEN SA.MaxBEpidosi
		  WHEN SA.MaxBEpidosi IS NOT NULL AND 
			   PO.BEpidosiDate < SA.MaxBEpidosi		THEN SA.MaxBEpidosi
		   ELSE PO.BEpidosiDate
	END AS BEpidosiDate
    ,CASE WHEN SA.MaxRank = 6 THEN 'Yes'
		  ELSE COALESCE(PO.AuctionCaseFlag, 'No')
	END AS AuctionCaseFlag
    ,CASE WHEN SA.MaxRank = 6 THEN 'Yes'
	      ELSE COALESCE(PO.AuctionCIFFlag, 'No')
	END AS AuctionCIFFlag
    ,CASE WHEN SA.MaxRank = 6 AND IFNULL(PO.AuctionSource, '') = ''		THEN 'Qualco Activities'
	 	  WHEN SA.MaxRank = 6 AND PO.AuctionSource LIKE '%Qualco%'		THEN PO.AuctionSource
	 	  WHEN SA.MaxRank = 6 												THEN 'Qualco Activities & ' || PO.AuctionSource
	 	  ELSE PO.AuctionSource
	END AS AuctionSource
	,CASE WHEN PO.KatathesiFlag = 'Yes' OR SA.MaxFlagsRankExclAuct >= 1	THEN 'Yes'
		  ELSE NULL
	END KatathesiFlag
	,CASE WHEN PO.EkdosiFlag = 'Yes' OR SA.MaxFlagsRankExclAuct >= 2	THEN 'Yes'
		  ELSE NULL
	END EkdosiFlag
	,CASE WHEN PO.AEpidosiFlag = 'Yes' OR SA.MaxFlagsRankExclAuct >= 3	THEN 'Yes'
		  ELSE NULL
	END AEpidosiFlag
	,CASE WHEN PO.BEpidosiFlag = 'Yes' OR SA.MaxFlagsRankExclAuct >= 4	THEN 'Yes'
		  ELSE NULL
	END BEpidosiFlag
	,CASE WHEN PO.TelesidikiaDPFlag = 'Yes' OR SA.MaxFlagsRankExclAuct >= 5	THEN 'Yes'
		  ELSE NULL
	END TelesidikiaDPFlag
	,PO.ExistingInInternalFile
	,PO.Priority
FROM PaymentOrder_new AS PO
INNER JOIN StrongestAct	AS SA ON SA.ExposureKey = PO.ExposureKey
'''
)
Updated.createOrReplaceTempView("Updated")

In [0]:
%python
PaymentOrder_Updated = spark.sql(
f'''
SELECT * from Updated
UNION
SELECT * FROM PaymentOrder_new
WHERE ExposureKey not in (select ExposureKey from Updated)
'''
)
PaymentOrder_Updated.createOrReplaceTempView("PaymentOrder_Updated")

In [0]:
%python
PaymentOrder_final = spark.sql(
f'''
select 
       BusinessDate
      ,ExposureKey
      ,PortfolioName
      ,CDI
      ,AgreementKey
      ,AgreementKeyClean
      ,PartyTaxRegistrationNumber
      ,DebtorName
     ,Product
     ,SubProduct
     ,AssetSegment
     ,IsActiveFlag
     ,AccountStatus
     ,TotalBalanceEur
     ,PaymentOrderCurrentAgency
     ,DPStatus
     ,Source
     ,DPDate
     ,case when RecentEpidosiDate is null then DPDate
           else RecentEpidosiDate
     end as RecentEpidosiDate
     ,AEpidosiDate
     ,BEpidosiDate
     ,AuctionCaseFlag
     ,AuctionCIFFlag
     ,AuctionSource
     ,KatathesiFlag
     ,EkdosiFlag
     ,AEpidosiFlag
     ,BEpidosiFlag
     ,TelesidikiaDPFlag
     ,ExistingInInternalFile
     ,case when Source = 'Qualco' then '1.Qualco'
           else Priority
     end as Priority
from PaymentOrder_Updated
'''
)
PaymentOrder_final.createOrReplaceTempView("PaymentOrder_final")

##### PaymentOrder Final

In [0]:
%python
Core_PaymentOrder = spark.sql(
f'''
SELECT  distinct
			  PO.BusinessDate
			,	PO.PortfolioName
			,	PO.AgreementKey
			,	PO.AgreementKeyClean
			,	PO.ExposureKey
			,	PO.CDI
			,	CASE WHEN PO.PartyTaxRegistrationNumber IN ('0000','000000000', '999999999', '') THEN NULL ELSE PO.PartyTaxRegistrationNumber END AS PartyTaxRegistrationNumber	
			,	po.DebtorName
			,	PO.Product
			,	PO.SubProduct
			,	PO.AssetSegment
			,	PO.IsActiveFlag
			,	PO.AccountStatus
			,	PO.TotalBalanceEur 
			,	po.PaymentOrderCurrentAgency
			,	PO.DPStatus
			,	PO.Source
			,	PO.DPDate
			,	PO.RecentEpidosiDate
			,	PO.AEpidosiDate
			,	PO.BEpidosiDate
			,	PO.AuctionCaseFlag
			,	PO.AuctionCIFFlag
			,	PO.AuctionSource
			,	PO.KatathesiFlag
			,	PO.EkdosiFlag
			,	PO.AEpidosiFlag
			,	PO.BEpidosiFlag
			,	PO.TelesidikiaDPFlag
			,	PO.ExistingInInternalFile
			,	PO.Priority
			,	SA.KatathesiMaxDate AS KatathesiMaxDateQC
			,	SA.EkdosiMaxDate    AS EkdosiMaxDateQC
			,	SA.MaxAEpidosi	    AS AEpidosiMaxDateQC
			,	SA.MaxBEpidosi	    AS BEpidosiMaxDateQC
		FROM PaymentOrder_final	  AS PO
		LEFT JOIN StrongestAct AS SA ON PO.ExposureKey = SA.ExposureKey
		LEFT JOIN NonCorePerimeter ncp 
		-- LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.int_NonCorePerimeter ncp 
		on ncp.Exposurekey = PO.ExposureKey
		where ncp.Exposurekey is null
		--where PO.IsActiveFlag = true
'''
)
Core_PaymentOrder.createOrReplaceTempView("Core_PaymentOrder")

##### final_df

In [0]:
%python
final_df = spark.sql(
f'''
SELECT  distinct
			  cast(BusinessDate as date) AS BusinessDate
			,	cast(PortfolioName as string) AS PortfolioName
			,	cast(AgreementKey as string) AS AgreementKey
			,	cast(AgreementKeyClean as string) AS AgreementKeyClean
			,	cast(ExposureKey as string) AS ExposureKey
			,	cast(CDI as string) AS CDI
			,	cast(PartyTaxRegistrationNumber as string) AS PartyTaxRegistrationNumber	
			,	cast(DebtorName as string) AS DebtorName
			,	cast(Product as string) AS Product
			,	cast(SubProduct as string) AS SubProduct
			,	cast(AssetSegment as string) AS AssetSegment
			,	cast(IsActiveFlag as boolean) AS IsActiveFlag
			,	cast(AccountStatus as string) AS AccountStatus
			,	cast(TotalBalanceEur as decimal(19,2)) AS TotalBalanceEur 
			,	cast(PaymentOrderCurrentAgency as string) AS PaymentOrderCurrentAgency
			,	cast(DPStatus as string) AS DPStatus
			,	cast(Source as string) AS Source
			,	cast(DPDate as date) AS DPDate
			,	cast(RecentEpidosiDate as date) AS RecentEpidosiDate
			,	cast(AEpidosiDate as date) AS AEpidosiDate
			,	cast(BEpidosiDate as date) AS BEpidosiDate
			,	cast(AuctionCaseFlag as string) AS AuctionCaseFlag
			,	cast(AuctionCIFFlag as string) AS AuctionCIFFlag
			,	cast(AuctionSource as string) AS AuctionSource
			,	cast(KatathesiFlag as string) AS KatathesiFlag
			,	cast(EkdosiFlag as string) AS EkdosiFlag
			,	cast(AEpidosiFlag as string) AS AEpidosiFlag
			,	cast(BEpidosiFlag as string) AS BEpidosiFlag
			,	cast(TelesidikiaDPFlag as string) AS TelesidikiaDPFlag
			,	cast(ExistingInInternalFile as string) AS ExistingInInternalFile
			,	cast(Priority as string) AS Priority
			,	cast(KatathesiMaxDateQC as date) AS KatathesiMaxDateQC
			,	cast(EkdosiMaxDateQC as date) AS EkdosiMaxDateQC
			,	cast(AEpidosiMaxDateQC as date) AS AEpidosiMaxDateQC
			,	cast(BEpidosiMaxDateQC as date) AS BEpidosiMaxDateQC
			,	cast(getdate() as timestamp) AS InsertTimeStamp
		FROM Core_PaymentOrder 
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_PaymentOrder: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = []
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
PaymentOrder = spark.sql(
f'''
SELECT * 
FROM {delivery_table_path}
where BusinessDate = {run_date!r} 
'''
)
PaymentOrder.createOrReplaceTempView("PaymentOrder")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/PaymentOrder/"
file_name = "PaymentOrder_" +run_date+ ".csv"
PaymentOrder.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)

In [0]:
payment_order_abexport = spark.sql(
f'''
SELECT * 
FROM {delivery_table_path}
where BusinessDate = {run_date!r}
and isActiveFlag = true
'''
)
payment_order_abexport.createOrReplaceTempView("payment_order_abexport")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/PaymentOrder_ABExport/"
file_name = "PaymentOrder_ABExport_" +run_date+ ".csv"
payment_order_abexport.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)